In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Mount Google Drive first:
# from google.colab import drive
# drive.mount('/content/drive')

BASE_PATH  = "/content/drive/MyDrive/ADNI1/processed"
IMG_SIZE   = (160, 160)
BATCH_SIZE = 32
EPOCHS_CNN = 10
EPOCHS_RES = 20
CLASSES    = ["AD", "CN", "MCI"]
NUM_CLASSES = 3

TRAIN_DIR = os.path.join(BASE_PATH, "train")
VAL_DIR   = os.path.join(BASE_PATH, "val")
TEST_DIR  = os.path.join(BASE_PATH, "test")

In [ ]:
print("Dataset counts:")
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    total = 0
    for cls in CLASSES:
        count = len(os.listdir(os.path.join(BASE_PATH, split, cls)))
        print(f"  {cls}: {count}")
        total += count
    print(f"  Total: {total}")

Dataset counts:

TRAIN
  AD: 9360
  CN: 12240
  MCI: 19920
  Total: 41520

VAL
  AD: 2220
  CN: 2940
  MCI: 4680
  Total: 9840

TEST
  AD: 2280
  CN: 2940
  MCI: 4620
  Total: 9840


In [ ]:
train_labels = []
for idx, cls in enumerate(CLASSES):
    folder = os.path.join(TRAIN_DIR, cls)
    train_labels.extend([idx] * len(os.listdir(folder)))

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=np.array(train_labels)
)
class_weight_dict = {0: class_weights[0],
                     1: class_weights[1],
                     2: class_weights[2]}
print("\nClass weights:", class_weight_dict)



Class weights: {0: np.float64(1.4786324786324787), 1: np.float64(1.130718954248366), 2: np.float64(0.6947791164658634)}


In [ ]:
# Training generator WITH augmentation (Unit 2: Regularization)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,       # brain is roughly symmetric
    rotation_range=10,          # small rotation
    zoom_range=0.05,            # very conservative zoom
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

# Validation and test generators — NO augmentation
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False
)

print("\nClass indices:", train_generator.class_indices)

Found 41520 images belonging to 3 classes.
Found 9840 images belonging to 3 classes.
Found 9840 images belonging to 3 classes.

Class indices: {'AD': 0, 'CN': 1, 'MCI': 2}


In [ ]:
def build_simple_cnn(input_shape=(160, 160, 3), num_classes=3):
    model = models.Sequential([
        # Conv Block 1 (Unit 1: Convolution + ReLU)
        layers.Conv2D(32, (3, 3), activation='relu',
                      input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),          # Unit 4: Batch Normalization

        # Conv Block 2
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),

        # Pooling (Unit 1: Downsampling)
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),                # Unit 2: Dropout Regularization

        # Conv Block 3
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),

        # Conv Block 4
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),

        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),

        # Fully Connected (Unit 1: Classification layer)
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),                 # Unit 2: Dropout
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = build_simple_cnn()
cnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 160, 160, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 160, 160, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 80, 80, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 80, 80, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 102400)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    26,214,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,282,787 (100.26 MB)

 Trainable params: 26,281,891 (100.26 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
cnn_callbacks = [
    # Unit 2: Early Stopping — prevents overfitting
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when stuck
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Save best model
    callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/ADNI1/models/best_cnn.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

os.makedirs('/content/drive/MyDrive/ADNI1/models', exist_ok=True)

# ── Cell 9: Train Simple CNN ─────────────────────────────────
print("\nTraining Simple CNN...")
cnn_history = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    class_weight=class_weight_dict,   # Unit 2: handles class imbalance
    callbacks=cnn_callbacks,
    verbose=1
)


Training Simple CNN...
Epoch 1/10
 278/1298 ━━━━━━━━━━━━━━━━━━━━ 5:26:16 19s/step - accuracy: 0.3599 - loss: 1.5698